# Notebook 5 — Reconstrucción sparse mediante ISTA y FISTA

## Objetivo

En este notebook se estudia la reconstrucción de señales sparse a partir de medidas lineales incompletas mediante métodos de optimización con regularización \(\ell_1\).

Los objetivos principales son los siguientes:

1. formular el problema de reconstrucción sparse como un problema de optimización regularizada;
2. implementar el algoritmo ISTA;
3. implementar el algoritmo FISTA como versión acelerada de ISTA;
4. comparar ambos métodos en términos de reconstrucción y convergencia.

Este notebook resulta fundamental dentro del TFG porque completa el bloque algorítmico de *Compressed Sensing* y permite contrastar los métodos voraces, como OMP, con métodos basados en optimización proximal.

In [2]:
"""la reconstrucción sparse no solo puede hacerse con métodos voraces, sino también con métodos de optimización convexa/proximal."""
"""Tengo que pedirle a chat que me escriba bien aqui la ecuacion de la siguiente celda que no la sé mostrar bien"""

'Tengo que pedirle a chat que me escriba bien aqui la ecuacion de la siguiente celda que no la sé mostrar bien'

## Qué se está haciendo realmente en este notebook

En lugar de buscar directamente una solución sparse mediante selección iterativa de columnas, como ocurría con OMP, aquí se plantea el problema como una minimización de la forma

$$
\min_x \frac{1}{2}\|Ax-y\|_2^2 + \lambda \|x\|_1.
$$

El primer término fuerza que la solución sea consistente con las medidas observadas, mientras que el segundo favorece la sparsidad. Los algoritmos ISTA y FISTA permiten resolver este problema mediante iteraciones sucesivas que combinan pasos de gradiente y operaciones de umbralización suave.

In [3]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# Configuración gráfica
plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["font.size"] = 11

## 1. Funciones auxiliares

Se definen funciones para:

- generar señales sparse sintéticas;
- construir matrices de medida aleatorias;
- calcular métricas de error;
- aplicar el operador de umbralización suave.

In [4]:
def generate_sparse_signal(N, k, seed=None):
    """
    Genera una señal k-sparse de dimensión N.
    """
    rng = np.random.default_rng(seed)
    x = np.zeros(N)

    support = rng.choice(N, size=k, replace=False)
    values = rng.normal(loc=0.0, scale=1.0, size=k)

    x[support] = values
    return x, np.sort(support)


def generate_measurement_matrix(m, N, seed=None):
    """
    Genera una matriz gaussiana aleatoria (m, N) y normaliza sus columnas.
    """
    rng = np.random.default_rng(seed)
    A = rng.normal(size=(m, N))

    norms = np.linalg.norm(A, axis=0)
    norms[norms == 0] = 1.0
    A = A / norms

    return A


def mse(x, x_hat):
    return np.mean((x - x_hat) ** 2)


def relative_error(x, x_hat):
    denom = np.linalg.norm(x)
    if denom == 0:
        return 0.0
    return np.linalg.norm(x - x_hat) / denom


def soft_threshold(z, tau):
    """
    Operador de umbralización suave:
    S_tau(z) = sign(z) * max(|z| - tau, 0)
    """
    return np.sign(z) * np.maximum(np.abs(z) - tau, 0.0)

## 2. Función objetivo

Para seguir la convergencia de los algoritmos se define la función objetivo

$$
F(x) = \frac{1}{2}\|Ax-y\|_2^2 + \lambda \|x\|_1
$$

Esto permitirá estudiar no solo la calidad final de la reconstrucción, sino también cómo evoluciona cada método a lo largo de las iteraciones.